In [2]:
import pandas as pd
import random

In [3]:
data=pd.read_csv('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain/DataCoSupplyChainDataset.csv',encoding='latin1')
#print(data.head(5))
print(data.columns)

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id', 'order date',
       'Order Id', 'Order Item Cardprod Id', 'Order Item Discount',
       'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price',
       'Order Item Profit Ratio', 'Order Item Quantity', 'Sales',
       'Order Item Total', 'Order Profit Per Order', 'Order Region',
       'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id',
       'Product Category Id', 'Product Description', 'Product

In [4]:
# corrupt order & shipping date
def corrupt_date(val):
    try:
        dt=pd.to_datetime(val)
    except Exception:
        return val
    
    fmt=random.randint(0,5)
    if fmt==0: return dt.strftime("%m/%d/%Y")          # 01/13/2018(original-ish)
    if fmt==1: return dt.strftime("%d-%m-%Y")          # 13-01-2018
    if fmt==2: return dt.strftime("%Y/%m/%d")          # 2018/01/13
    if fmt==3: return dt.strftime("%B %d, %Y")         # January 13,2018
    if fmt==4: return dt.strftime("%d %b %Y")          # 13 Jan 2018
    return dt.strftime("%Y%m%d")                # 20180113

In [5]:
def corrupt_pct(data,pct):
    return data.sample(frac=pct).index

In [6]:
order_col="order date"
ship_col="shipping date"
if order_col in data.columns and ship_col in data.columns:
    idx_order=corrupt_pct(data,0.7)
    idx_ship=corrupt_pct(data,0.7)
    data.loc[idx_order,order_col]=data.loc[idx_order,order_col].apply(corrupt_date)
    data.loc[idx_ship,ship_col]=data.loc[idx_ship,ship_col].apply(corrupt_date)

In [7]:
#print(data[['order date','shipping date']].head(5))

In [8]:
# curropt name
salutations=["MS.","MRS.","MR.",""]
def corrupt_name(fname,lname):
    try:
        f=str(fname).strip()
        l=str(lname).strip()
    except Exception:
        return f"{fname} {lname}"
    
    sal=random.choice(salutations)
    fmt=random.randint(0,4)
    if fmt==0: return f"{sal}{f} {l}" #MS.Bhoomi Bhatt
    if fmt==1: return f"{sal}{f.upper()} {l}" #MS.BHOOMI bhatt
    if fmt==2: return f"{f} {l.upper()}" #Bhoomi BHATT
    if fmt==3: return f"{f.lower()} {l.lower()}" #bhoomi bhatt
    return f"{f}, {l}" #bhOOmi, bHatt


In [9]:
data["Customer Name"] = data.apply(
    lambda r: corrupt_name(r["Customer Fname"], r["Customer Lname"]), axis=1
)

In [10]:
#dropping columns
data.drop(columns=['Customer Fname','Customer Lname'],inplace=True)

In [11]:
# Corrupt delivery status
delivery_variants={
    "Late delivery":     ["Late delivery", "LATE DELIVERY", "late delivery", "Late Delivery", "LATE"],
    "Advance shipping":  ["Advance shipping", "advance shipping", "ADVANCE SHIPPING", "Advance Ship"],
    "Shipping on time":  ["Shipping on time", "shipping on time", "ON TIME", "On Time", "on time"],
    "Shipping canceled": ["Shipping canceled", "SHIPPING CANCELED", "Cancelled", "CANCELLED", "canceled"],
}

def corrupt_delivery_status(val):
    val=str(val).strip()
    for canonical,variants in delivery_variants.items():
        if val.lower() == canonical.lower() or val in variants:
            return random.choice(variants)
    return val

if "Delivery Status" in data.columns:
    idx=corrupt_pct(data,0.55)
    data.loc[idx,'Delivery Status']=data.loc[idx,'Delivery Status'].apply(corrupt_delivery_status)


In [12]:
#corrupt product price
def corrupt_price(val):
    try:
        v=float(val)
    except Exception:
        return val
    
    fmt=random.randint(0,3)
    if fmt==0: return f"${v:.2f}" #$327.34
    if fmt==1: return f"USD {v:.2f}" #USD 327.34
    if fmt==2: return f"{v:.2f}USD" #327.34USD
    return f"{v}"

if 'Product Price' in data.columns:
    idx=corrupt_pct(data,0.65)
    data.loc[idx,'Product Price']=data.loc[idx,'Product Price'].apply(corrupt_price)

/var/folders/lx/vxt9_2cx3x3b108zq_xn9djr0000gn/T/ipykernel_53836/745830108.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['129.9900055' '$24.99' '50.00USD' ... '99.99USD' '99.98999786' '$50.00']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data.loc[idx,'Product Price']=data.loc[idx,'Product Price'].apply(corrupt_price)


In [13]:
#adding duplicate rows
print("Dataset size before adding duplicates(row*column)",data.shape)
duplicate_idx=corrupt_pct(data,0.11)
dups=data.iloc[duplicate_idx].copy()
data=pd.concat([data,dups],ignore_index=True)
print("Dataset size after adding duplicates(row*column)",data.shape)

Dataset size before adding duplicates(row*column) (180519, 52)
Dataset size after adding duplicates(row*column) (200376, 52)


In [14]:
print(data.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Id', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping date', 'Shipping Mode', 'Customer Name']


In [15]:
#split dataset by yeat
data["year"] = pd.to_datetime(
    data[order_col], infer_datetime_format=True, errors="coerce").dt.year

for year, group in data.groupby("year"):
    group = group.drop(columns=["year"])
    fname = f"supplychain_{int(year)}.csv"
    group.to_csv(fname, index=False)
    print(f"{fname} → {len(group):,} rows")

/var/folders/lx/vxt9_2cx3x3b108zq_xn9djr0000gn/T/ipykernel_53836/862779502.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data["year"] = pd.to_datetime(


supplychain_2015.csv → 8,179 rows
supplychain_2016.csv → 8,001 rows
supplychain_2017.csv → 6,922 rows
supplychain_2018.csv → 268 rows


In [ ]:
supplychain_2016=pd.read_csv('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2016.csv')
supplychain_2016.to_parquet('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2016.parquet')

supplychain_2017=pd.read_csv('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2017.csv')
supplychain_2017.to_json('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2017.json')

In [24]:
pip install pandavro


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandavro as pdx

In [28]:
supplychain_2018 = pd.read_csv('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2018.csv', encoding='latin1')
pdx.to_avro('/Users/bhoomi/Documents/GitHub/ChainForge/supplychain_2018.avro', supplychain_2018)